In [ ]:
from qiskit import transpile
from qiskit.circuit import ParameterVector
from qiskit.transpiler import Target, InstructionProperties
from qiskit.circuit.library import RZGate, SXGate, XGate, Measure, ECRGate

## Compare transpilation on a custom `Target`

A `Target` lets us define a backend-like ISA ourselves, including:
- supported instructions,
- coupling constraints,
- durations,
- and **error rates**.

This is useful for:
- sensitivity studies,
- teaching,
- prototyping custom hardware assumptions,
- and explaining why the transpiler chooses certain decompositions.


In [ ]:
def make_custom_target(
    n_qubits=4,
    ecr_error=0.01,
    sx_error=2e-4,
    x_error=2e-4,
    meas_error=0.02,
    ecr_duration=5.0e-7,
    sx_duration=3.5e-8,
    x_duration=3.5e-8,
    meas_duration=1.2e-6,
):
    theta = ParameterVector("θ_target", 1)[0]
    target = Target(num_qubits=n_qubits)

    target.add_instruction(
        RZGate(theta),
        {(q,): None for q in range(n_qubits)},
        name="rz",
    )
    target.add_instruction(
        SXGate(),
        {(q,): InstructionProperties(duration=sx_duration, error=sx_error) for q in range(n_qubits)},
        name="sx",
    )
    target.add_instruction(
        XGate(),
        {(q,): InstructionProperties(duration=x_duration, error=x_error) for q in range(n_qubits)},
        name="x",
    )
    target.add_instruction(
        Measure(),
        {(q,): InstructionProperties(duration=meas_duration, error=meas_error) for q in range(n_qubits)},
        name="measure",
    )
    target.add_instruction(
        ECRGate(),
        {
            (0, 1): InstructionProperties(duration=ecr_duration, error=ecr_error),
            (1, 2): InstructionProperties(duration=ecr_duration, error=ecr_error),
            (2, 3): InstructionProperties(duration=ecr_duration, error=ecr_error),
        },
        name="ecr",
    )
    return target

target_low_noise = make_custom_target(ecr_error=0.005, meas_error=0.01)
target_high_noise = make_custom_target(ecr_error=0.03, meas_error=0.05)

isa_low = transpile(ansatz, target=target_low_noise, optimization_level=3, seed_transpiler=1234)
isa_high = transpile(ansatz, target=target_high_noise, optimization_level=3, seed_transpiler=1234)

summary = {
    "abstract_depth": ansatz.decompose().depth(),
    "isa_low_depth": isa_low.depth(),
    "isa_high_depth": isa_high.depth(),
    "isa_low_ops": dict(isa_low.count_ops()),
    "isa_high_ops": dict(isa_high.count_ops()),
}
summary


In [ ]:
isa_low.draw("mpl", idle_wires=False)

print("Low-noise custom target:")
for inst, qargs in target_low_noise.instructions:
    props = target_low_noise[inst.name].get(qargs, None)
    if props is not None and getattr(props, "error", None) is not None:
        print(f"{inst.name:8s} on {qargs}: error={props.error:.4g}, duration={props.duration}")


### Discussion

The transpiled circuit may be structurally identical for the two custom targets, because **Target error rates influence interpretation and benchmarking**, while the preset transpiler still primarily optimizes for a valid ISA implementation and circuit structure. In practice, custom cost models or backend-specific passes can be layered on top.


### Exercise

Change the custom target so that only the edge `(0, 1)` has low two-qubit error and the other edges are much worse.

Then inspect whether:
- the transpiler changes routing,
- the circuit depth changes,
- or only your *performance expectations* change.
